# Sequence Models

## Learning Objectives
- Understand Recurrent Neural Networks (RNNs) and their limitations
- Master LSTM and GRU architectures
- Learn about Transformers and attention mechanisms
- Apply sequence models to NLP and time series problems

## 5.1 Introduction to Sequence Models

Sequence models are designed to handle data where order matters:
- **Natural Language**: Sentences, documents
- **Time Series**: Stock prices, weather data
- **Video**: Sequences of images
- **Audio**: Speech signals

### Key Challenges:
- Variable length sequences
- Long-term dependencies
- Temporal patterns

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

print("Libraries imported successfully!")

## 5.2 Recurrent Neural Networks (RNNs)

In [ ]:
# Simple RNN implementation from scratch
class SimpleRNN:
    def __init__(self, input_size, hidden_size, output_size):
        self.hidden_size = hidden_size
        
        # Initialize weights
        self.Wxh = np.random.randn(input_size, hidden_size) * 0.01  # Input to hidden
        self.Whh = np.random.randn(hidden_size, hidden_size) * 0.01  # Hidden to hidden
        self.Why = np.random.randn(hidden_size, output_size) * 0.01  # Hidden to output
        
        self.bh = np.zeros((1, hidden_size))  # Hidden bias
        self.by = np.zeros((1, output_size))  # Output bias
    
    def forward(self, inputs):
        """Forward pass through RNN"""
        h = np.zeros((1, self.hidden_size))
        hidden_states = [h]
        outputs = []
        
        for x in inputs:
            # Reshape input if needed
            if len(x.shape) == 1:
                x = x.reshape(1, -1)
            
            # RNN forward step
            h = np.tanh(np.dot(x, self.Wxh) + np.dot(h, self.Whh) + self.bh)
            y = np.dot(h, self.Why) + self.by
            
            hidden_states.append(h.copy())
            outputs.append(y)
        
        return outputs, hidden_states

# Test the simple RNN
input_size = 3
hidden_size = 4
output_size = 2
sequence_length = 5

# Create sample input sequence
sample_inputs = [np.random.randn(input_size) for _ in range(sequence_length)]

rnn = SimpleRNN(input_size, hidden_size, output_size)
outputs, hidden_states = rnn.forward(sample_inputs)

print(f"Input sequence length: {len(sample_inputs)}")
print(f"Output sequence length: {len(outputs)}")
print(f"Hidden state shape: {hidden_states[0].shape}")
print(f"Output shape: {outputs[0].shape}")

# Visualize hidden states evolution
hidden_states_array = np.array(hidden_states).squeeze()
plt.figure(figsize=(12, 8))

for i in range(hidden_size):
    plt.plot(hidden_states_array[:, i], label=f'Hidden {i+1}')

plt.title('Hidden States Evolution Over Time')
plt.xlabel('Time Step')
plt.ylabel('Hidden State Value')
plt.legend()
plt.grid(True)
plt.show()

## 5.3 LSTM Networks

In [ ]:
# Generate synthetic time series data
def generate_time_series(n_samples=1000, n_timesteps=50, noise_level=0.1):
    """Generate synthetic time series with patterns"""
    t = np.linspace(0, 4*np.pi, n_timesteps)
    
    series = []
    labels = []
    
    for i in range(n_samples):
        # Create different patterns
        pattern_type = np.random.choice(['sine', 'cosine', 'combined'])
        
        if pattern_type == 'sine':
            base = np.sin(t + np.random.uniform(0, 2*np.pi))
            label = 0
        elif pattern_type == 'cosine':
            base = np.cos(t + np.random.uniform(0, 2*np.pi))
            label = 1
        else:  # combined
            base = 0.5 * np.sin(t) + 0.5 * np.cos(2*t)
            label = 2
        
        # Add noise
        noisy_series = base + np.random.normal(0, noise_level, n_timesteps)
        
        series.append(noisy_series)
        labels.append(label)
    
    return np.array(series), np.array(labels)

# Generate data
X_series, y_series = generate_time_series(n_samples=2000, n_timesteps=50, noise_level=0.2)

# Reshape for LSTM (samples, timesteps, features)
X_series = X_series.reshape((X_series.shape[0], X_series.shape[1], 1))

# Split data
X_train_series, X_test_series, y_train_series, y_test_series = train_test_split(
    X_series, y_series, test_size=0.2, random_state=42, stratify=y_series
)

print(f"Training data shape: {X_train_series.shape}")
print(f"Test data shape: {X_test_series.shape}")
print(f"Class distribution: {np.bincount(y_series)}")

# Visualize sample sequences
plt.figure(figsize=(15, 5))
for i in range(3):
    plt.subplot(1, 3, i+1)
    mask = y_train_series == i
    sample_idx = np.where(mask)[0][0]
    plt.plot(X_train_series[sample_idx, :, 0])
    plt.title(f'Class {i} Sample')
    plt.xlabel('Time Step')
    plt.ylabel('Value')
    plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Build LSTM model
def build_lstm_model(input_shape, num_classes, lstm_units=64, dropout_rate=0.2):
    """Build LSTM model for sequence classification"""
    model = keras.Sequential([
        layers.LSTM(lstm_units, return_sequences=True, input_shape=input_shape),
        layers.Dropout(dropout_rate),
        layers.LSTM(lstm_units//2, return_sequences=False),
        layers.Dropout(dropout_rate),
        layers.Dense(32, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Build and train LSTM model
lstm_model = build_lstm_model(
    input_shape=(X_train_series.shape[1], X_train_series.shape[2]),
    num_classes=3,
    lstm_units=64,
    dropout_rate=0.3
)

lstm_model.summary()

# Train the model
print("\nTraining LSTM model...")
history_lstm = lstm_model.fit(
    X_train_series, y_train_series,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

In [ ]:
# Evaluate LSTM model
test_loss, test_acc = lstm_model.evaluate(X_test_series, y_test_series, verbose=0)
print(f"LSTM Test Accuracy: {test_acc:.4f}")

# Make predictions
y_pred_proba = lstm_model.predict(X_test_series)
y_pred = np.argmax(y_pred_proba, axis=1)

# Classification report
print("\nClassification Report:")
print(classification_report(y_test_series, y_pred))

# Plot training history
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(history_lstm.history['loss'], label='Training Loss')
plt.plot(history_lstm.history['val_loss'], label='Validation Loss')
plt.title('LSTM Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(history_lstm.history['accuracy'], label='Training Accuracy')
plt.plot(history_lstm.history['val_accuracy'], label='Validation Accuracy')
plt.title('LSTM Training Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Confusion matrix
plt.subplot(1, 3, 3)
cm = confusion_matrix(y_test_series, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Sine', 'Cosine', 'Combined'], 
            yticklabels=['Sine', 'Cosine', 'Combined'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')

plt.tight_layout()
plt.show()

## 5.4 GRU Networks

In [ ]:
# Build GRU model
def build_gru_model(input_shape, num_classes, gru_units=64, dropout_rate=0.2):
    """Build GRU model for sequence classification"""
    model = keras.Sequential([
        layers.GRU(gru_units, return_sequences=True, input_shape=input_shape),
        layers.Dropout(dropout_rate),
        layers.GRU(gru_units//2, return_sequences=False),
        layers.Dropout(dropout_rate),
        layers.Dense(32, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Build and train GRU model
gru_model = build_gru_model(
    input_shape=(X_train_series.shape[1], X_train_series.shape[2]),
    num_classes=3,
    gru_units=64,
    dropout_rate=0.3
)

print("Training GRU model...")
history_gru = gru_model.fit(
    X_train_series, y_train_series,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

# Evaluate GRU model
test_loss_gru, test_acc_gru = gru_model.evaluate(X_test_series, y_test_series, verbose=0)
print(f"GRU Test Accuracy: {test_acc_gru:.4f}")

# Compare LSTM vs GRU
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.plot(history_lstm.history['val_accuracy'], label='LSTM')
plt.plot(history_gru.history['val_accuracy'], label='GRU')
plt.title('Validation Accuracy Comparison')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(history_lstm.history['val_loss'], label='LSTM')
plt.plot(history_gru.history['val_loss'], label='GRU')
plt.title('Validation Loss Comparison')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 3)
models = ['LSTM', 'GRU']
accuracies = [test_acc, test_acc_gru]
bars = plt.bar(models, accuracies, color=['blue', 'orange'])
plt.title('Final Test Accuracy Comparison')
plt.ylabel('Accuracy')
plt.ylim(0, 1)

for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{acc:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 5.5 Text Processing with Sequence Models

In [ ]:
# Sample text data for sentiment analysis
texts = [
    "This movie was fantastic! I loved every minute of it.",
    "The acting was terrible and the plot was boring.",
    "I'm not sure how I feel about this film.",
    "Amazing cinematography and great performances.",
    "Worst movie I've ever seen. Complete waste of time.",
    "Pretty good, but could have been better.",
    "Incredible storyline and character development.",
    "I fell asleep halfway through the movie.",
    "Brilliant! A masterpiece of modern cinema.",
    "Not bad, but not great either."
]

# Labels (0: negative, 1: neutral, 2: positive)
labels = [2, 0, 1, 2, 0, 1, 2, 0, 2, 1]

# Create more samples by variations
augmented_texts = []
augmented_labels = []

for text, label in zip(texts, labels):
    augmented_texts.append(text)
    augmented_labels.append(label)
    
    # Add simple variations
    if label == 2:  # Positive
        augmented_texts.append(text.replace("fantastic", "excellent"))
        augmented_texts.append(text.replace("amazing", "wonderful"))
        augmented_labels.extend([2, 2])
    elif label == 0:  # Negative
        augmented_texts.append(text.replace("terrible", "awful"))
        augmented_texts.append(text.replace("worst", "terrible"))
        augmented_labels.extend([0, 0])
    else:  # Neutral
        augmented_texts.append(text.replace("not sure", "uncertain"))
        augmented_texts.append(text.replace("not bad", "okay"))
        augmented_labels.extend([1, 1])

print(f"Total samples: {len(augmented_texts)}")
print(f"Class distribution: {np.bincount(augmented_labels)}")

# Tokenize text
tokenizer = Tokenizer(num_words=1000, oov_token="<OOV>")
tokenizer.fit_on_texts(augmented_texts)

# Convert text to sequences
sequences = tokenizer.texts_to_sequences(augmented_texts)

# Pad sequences
max_length = max(len(seq) for seq in sequences)
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')

print(f"\nVocabulary size: {len(tokenizer.word_index)}")
print(f"Max sequence length: {max_length}")
print(f"Padded sequences shape: {padded_sequences.shape}")

# Show some examples
print("\nSample tokenization:")
for i in range(3):
    print(f"Text: {augmented_texts[i]}")
    print(f"Sequence: {sequences[i]}")
    print(f"Padded: {padded_sequences[i]}")
    print()

In [ ]:
# Split text data
X_train_text, X_test_text, y_train_text, y_test_text = train_test_split(
    padded_sequences, np.array(augmented_labels), 
    test_size=0.2, random_state=42, stratify=np.array(augmented_labels)
)

# Build text classification model
def build_text_model(vocab_size, embedding_dim, max_length, num_classes):
    """Build LSTM model for text classification"""
    model = keras.Sequential([
        layers.Embedding(vocab_size, embedding_dim, input_length=max_length),
        layers.LSTM(64, return_sequences=True),
        layers.Dropout(0.3),
        layers.LSTM(32),
        layers.Dropout(0.3),
        layers.Dense(16, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Build and train text model
text_model = build_text_model(
    vocab_size=len(tokenizer.word_index) + 1,
    embedding_dim=50,
    max_length=max_length,
    num_classes=3
)

text_model.summary()

# Train the model
print("\nTraining text classification model...")
history_text = text_model.fit(
    X_train_text, y_train_text,
    epochs=100,
    batch_size=8,
    validation_split=0.2,
    verbose=0
)

# Evaluate
test_loss_text, test_acc_text = text_model.evaluate(X_test_text, y_test_text, verbose=0)
print(f"Text Classification Test Accuracy: {test_acc_text:.4f}")

## 5.6 Introduction to Transformers

In [ ]:
# Simplified Self-Attention mechanism
class SelfAttention:
    def __init__(self, embed_dim):
        self.embed_dim = embed_dim
        # Initialize weight matrices
        self.Wq = np.random.randn(embed_dim, embed_dim) * 0.01
        self.Wk = np.random.randn(embed_dim, embed_dim) * 0.01
        self.Wv = np.random.randn(embed_dim, embed_dim) * 0.01
    
    def forward(self, x):
        """Forward pass of self-attention"""
        # x shape: (seq_len, embed_dim)
        
        # Compute Q, K, V
        Q = np.dot(x, self.Wq)  # (seq_len, embed_dim)
        K = np.dot(x, self.Wk)  # (seq_len, embed_dim)
        V = np.dot(x, self.Wv)  # (seq_len, embed_dim)
        
        # Compute attention scores
        scores = np.dot(Q, K.T) / np.sqrt(self.embed_dim)  # (seq_len, seq_len)
        
        # Apply softmax
        attention_weights = self.softmax(scores)
        
        # Compute output
        output = np.dot(attention_weights, V)  # (seq_len, embed_dim)
        
        return output, attention_weights
    
    def softmax(self, x):
        """Softmax function"""
        exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
        return exp_x / np.sum(exp_x, axis=1, keepdims=True)

# Demonstrate self-attention
seq_len = 5
embed_dim = 4

# Create sample input (sequence of word embeddings)
sample_input = np.random.randn(seq_len, embed_dim)

attention = SelfAttention(embed_dim)
output, attention_weights = attention.forward(sample_input)

print(f"Input shape: {sample_input.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attention_weights.shape}")

# Visualize attention weights
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(sample_input, cmap='viridis', aspect='auto')
plt.title('Input Embeddings')
plt.xlabel('Embedding Dimension')
plt.ylabel('Sequence Position')
plt.colorbar()

plt.subplot(1, 2, 2)
plt.imshow(attention_weights, cmap='Blues', aspect='auto')
plt.title('Attention Weights')
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.colorbar()

plt.tight_layout()
plt.show()

print("\nAttention Weights Matrix:")
print(attention_weights.round(3))

In [ ]:
# Multi-Head Attention (simplified)
class MultiHeadAttention:
    def __init__(self, embed_dim, num_heads):
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"
        
        # Initialize weight matrices for each head
        self.Wq = np.random.randn(embed_dim, embed_dim) * 0.01
        self.Wk = np.random.randn(embed_dim, embed_dim) * 0.01
        self.Wv = np.random.randn(embed_dim, embed_dim) * 0.01
        self.Wo = np.random.randn(embed_dim, embed_dim) * 0.01
    
    def forward(self, x):
        """Forward pass of multi-head attention"""
        batch_size, seq_len, embed_dim = x.shape
        
        # Compute Q, K, V
        Q = np.dot(x, self.Wq)  # (batch_size, seq_len, embed_dim)
        K = np.dot(x, self.Wk)  # (batch_size, seq_len, embed_dim)
        V = np.dot(x, self.Wv)  # (batch_size, seq_len, embed_dim)
        
        # Reshape for multi-head
        Q = Q.reshape(batch_size, seq_len, self.num_heads, self.head_dim)
        K = K.reshape(batch_size, seq_len, self.num_heads, self.head_dim)
        V = V.reshape(batch_size, seq_len, self.num_heads, self.head_dim)
        
        # Transpose for attention computation
        Q = Q.transpose(0, 2, 1, 3)  # (batch_size, num_heads, seq_len, head_dim)
        K = K.transpose(0, 2, 1, 3)
        V = V.transpose(0, 2, 1, 3)
        
        # Compute attention for each head
        head_outputs = []
        for i in range(self.num_heads):
            head_Q = Q[:, i, :, :]  # (batch_size, seq_len, head_dim)
            head_K = K[:, i, :, :]
            head_V = V[:, i, :, :]
            
            # Attention scores
            scores = np.dot(head_Q, head_K.transpose(0, 2, 1)) / np.sqrt(self.head_dim)
            attention_weights = self.softmax_batch(scores)
            
            # Head output
            head_output = np.dot(attention_weights, head_V)
            head_outputs.append(head_output)
        
        # Concatenate heads
        concat_output = np.stack(head_outputs, axis=2)  # (batch_size, seq_len, num_heads, head_dim)
        concat_output = concat_output.reshape(batch_size, seq_len, self.embed_dim)
        
        # Final linear projection
        output = np.dot(concat_output, self.Wo)
        
        return output
    
    def softmax_batch(self, x):
        """Softmax for batch of matrices"""
        exp_x = np.exp(x - np.max(x, axis=2, keepdims=True))
        return exp_x / np.sum(exp_x, axis=2, keepdims=True)

# Demonstrate multi-head attention
batch_size = 2
seq_len = 4
embed_dim = 8
num_heads = 4

sample_input_batch = np.random.randn(batch_size, seq_len, embed_dim)

multi_head_attention = MultiHeadAttention(embed_dim, num_heads)
mh_output = multi_head_attention.forward(sample_input_batch)

print(f"Input shape: {sample_input_batch.shape}")
print(f"Multi-head output shape: {mh_output.shape}")
print(f"Number of heads: {num_heads}")
print(f"Head dimension: {embed_dim // num_heads}")

## 5.7 Transformer Architecture Overview

In [ ]:
# Simplified Transformer block
class TransformerBlock:
    def __init__(self, embed_dim, num_heads, ff_dim):
        self.attention = MultiHeadAttention(embed_dim, num_heads)
        self.ff_dim = ff_dim
        
        # Feed-forward network weights
        self.W1 = np.random.randn(embed_dim, ff_dim) * 0.01
        self.b1 = np.zeros(ff_dim)
        self.W2 = np.random.randn(ff_dim, embed_dim) * 0.01
        self.b2 = np.zeros(embed_dim)
        
        # Layer normalization parameters
        self.ln1_gamma = np.ones(embed_dim)
        self.ln1_beta = np.zeros(embed_dim)
        self.ln2_gamma = np.ones(embed_dim)
        self.ln2_beta = np.zeros(embed_dim)
    
    def layer_norm(self, x, gamma, beta):
        """Layer normalization"""
        mean = np.mean(x, axis=-1, keepdims=True)
        var = np.var(x, axis=-1, keepdims=True)
        normalized = (x - mean) / np.sqrt(var + 1e-6)
        return gamma * normalized + beta
    
    def feed_forward(self, x):
        """Feed-forward network"""
        # First linear layer + ReLU
        hidden = np.maximum(0, np.dot(x, self.W1) + self.b1)
        # Second linear layer
        output = np.dot(hidden, self.W2) + self.b2
        return output
    
    def forward(self, x):
        """Forward pass of transformer block"""
        # Multi-head attention with residual connection
        attn_output = self.attention.forward(x)
        x1 = x + attn_output
        x1 = self.layer_norm(x1, self.ln1_gamma, self.ln1_beta)
        
        # Feed-forward network with residual connection
        ff_output = self.feed_forward(x1)
        x2 = x1 + ff_output
        x2 = self.layer_norm(x2, self.ln2_gamma, self.ln2_beta)
        
        return x2

# Demonstrate transformer block
embed_dim = 8
num_heads = 4
ff_dim = 16

transformer = TransformerBlock(embed_dim, num_heads, ff_dim)
transformer_output = transformer.forward(sample_input_batch)

print(f"Input shape: {sample_input_batch.shape}")
print(f"Transformer output shape: {transformer_output.shape}")

# Show the difference between input and output
print(f"\nInput sample (first sequence, first token): {sample_input_batch[0, 0, :].round(3)}")
print(f"Output sample (first sequence, first token): {transformer_output[0, 0, :].round(3)}")

# Visualize the transformation
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.imshow(sample_input_batch[0], cmap='viridis', aspect='auto')
plt.title('Input Sequence')
plt.xlabel('Embedding Dimension')
plt.ylabel('Sequence Position')
plt.colorbar()

plt.subplot(1, 3, 2)
plt.imshow(transformer_output[0], cmap='viridis', aspect='auto')
plt.title('Transformer Output')
plt.xlabel('Embedding Dimension')
plt.ylabel('Sequence Position')
plt.colorbar()

plt.subplot(1, 3, 3)
diff = transformer_output[0] - sample_input_batch[0]
plt.imshow(diff, cmap='RdBu', aspect='auto')
plt.title('Difference (Output - Input)')
plt.xlabel('Embedding Dimension')
plt.ylabel('Sequence Position')
plt.colorbar()

plt.tight_layout()
plt.show()

## 5.8 Key Takeaways

### Recurrent Neural Networks
- **RNNs** process sequences step by step
- **Vanilla RNNs** suffer from vanishing/exploding gradients
- **Hidden state** carries information through time

### LSTM and GRU
- **LSTM** uses gates to control information flow
- **GRU** is simpler but often performs similarly
- **Both** solve long-term dependency problems

### Transformers
- **Self-attention** allows parallel processing
- **Multi-head attention** captures different relationships
- **Positional encoding** adds sequence position information
- **Transformers** dominate modern NLP

### Applications
- **Time series**: Forecasting, anomaly detection
- **NLP**: Translation, sentiment analysis, text generation
- **Speech**: Recognition, synthesis
- **Video**: Action recognition, captioning

## Exercises

1. **RNN Variants**: Implement bidirectional RNNs
2. **Attention Visualization**: Create attention weight visualizations
3. **Sequence Generation**: Build a text generation model
4. **Time Series Forecasting**: Apply to real financial data
5. **Transformer Training**: Train a small transformer from scratch